In [1]:
#| default_exp g2p

# Text to phonemes, without Python

> Kokoro's front end on the phone: misaki in Rust, a build-time lexicon for the words it can't say, a small model for
> the words no lexicon has, and abbreviations written out first.

Notebook 106 put everything from phonemes to audio on the phone. What turns text into phonemes is misaki: Python,
leaning on spaCy to tokenize and tag, and on espeak-ng for any word not in its lexicons. None of that runs in an app,
and espeak-ng is GPLv3, which would make the whole app GPL. This notebook finds what misaki can't say, builds what
fills the gaps without shipping espeak, and checks the Rust that does it all in the apps, `~/audio-reader/native/g2p`.

It downloads data, trains for over an hour (once; later runs load the result) and builds Rust, so `nbdev-test` skips
it.

In [2]:
#| export
import functools
import glob
import json
import math
import os
import random
import re
import shutil
import string
import struct
import subprocess
import tarfile
import time
import urllib.request
from collections import Counter
from importlib.resources import files
from pathlib import Path

import torch
import torch.nn.functional as F
from misaki import en
from misaki.espeak import EspeakFallback
from misaki.token import MToken

from text_to_speech.core import DATA
from text_to_speech.data import metadata
from text_to_speech.phone import ANDROID, IOS, SIZES, VOICE, size_mb
from text_to_speech.voice import SR, model

scikit-learn version 1.9.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.


Torch version 2.15.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.8.0 is the most recent version that has been tested.


In [3]:
from text_to_speech.voice import pipeline

Both apps download the same text-to-phoneme files, in `data/voice/text`. The Rust lives in audio-reader.

In [4]:
#| export
TEXT = VOICE / "text"
G2P = DATA / "g2p"
NATIVE = Path(os.environ.get("MAGPIE_G2P", Path.home() / "audio-reader" / "native" / "g2p"))  # Or another checkout
TEXT.mkdir(parents=True, exist_ok=True)
G2P.mkdir(parents=True, exist_ok=True)

What the notebooks have used so far:

In [5]:
print(type(pipeline("af_heart").g2p).__module__, type(pipeline("af_heart").g2p.fallback).__name__)

misaki.en EspeakFallback


## What misaki can't say

A corpus to measure on: 1,500 random LJSpeech sentences (a 1960s government report and Victorian prose, heavy on
names) and every prose paragraph of audio-reader's own documentation (modern, technical, full of numbers).

In [6]:
#| export
def corpus(n=1500, seed=0):
    "LJSpeech sentences, and the prose paragraphs of audio-reader's docs."
    lines = list(metadata().normalized_text.dropna())
    random.seed(seed)
    lines = random.sample(lines, n)
    for path in sorted(
        glob.glob(str(Path.home() / "audio-reader" / "docs" / "*.md"))
        + glob.glob(str(Path.home() / "audio-reader" / "*.md"))
    ):
        for para in Path(path).read_text().split("\n\n"):
            para = " ".join(para.split())
            if len(para) > 60 and not para.startswith(("|", "#", "```", "-", "<", "*")) and "`" not in para:
                lines.append(para)
    return lines


def unknown(tokens):
    "The tokens with letters in them that misaki left without phonemes."
    return [t for t in tokens if any(c.isalpha() for c in t.text) and (t.phonemes is None or "❓" in t.phonemes)]

In [7]:
lines = corpus()
g2p = {accent: en.G2P(trf=False, british=accent == "gb", fallback=None) for accent in ["us", "gb"]}
python = {accent: [g(line) for line in lines] for accent, g in g2p.items()}
for accent, results in python.items():
    words = sum(1 for _, tokens in results for t in tokens if any(c.isalpha() for c in t.text))
    missing = Counter(t.text for _, tokens in results for t in unknown(tokens))
    print(
        f"{accent}: {len(lines)} lines, {words} words, {sum(missing.values())} unknown ({sum(missing.values()) / words:.1%})"
    )
    print("   ", ", ".join(w for w, _ in missing.most_common(20)))

us: 1974 lines, 49863 words, 713 unknown (1.4%)
    Oswald, Oswald's, testified, Datalab, Frazier, Xcode, Harvey, denied, Tippit, Paine, Jev, verified, Connally, Jefferson, Messrs., LPCNet, roominghouse, Hidell, De, palmprint
gb: 1974 lines, 49863 words, 731 unknown (1.5%)
    Oswald, Oswald's, testified, Kennedy, Datalab, Frazier, Xcode, Harvey, Tippit, Paine, Jev, verified, MiB, Connally, Jefferson, LPCNet, roominghouse, Hidell, De, palmprint


About one word in seventy. Mostly names, and some inflections misaki's stemming doesn't reach ("testified",
"verified"), but also "Kennedy" in British English. Kokoro's vocabulary has no symbol for ❓, so without a fallback
each of these words would simply be skipped: for a listener who can't see the text, the worst possible failure.

## A lexicon built in advance

Running espeak-ng on the phone would make the app GPL. Running it here, once, to write a lexicon of its
pronunciations is a different matter: the app ships data, not espeak. The words: everything in CMUdict (BSD licensed,
134,000 words, many of them names) and in macOS's word lists (Webster's Second, and a list of first names).

CMUdict is in ARPAbet. Mapped into misaki's phonemes, it matches misaki's own lexicon more often than espeak does:

In [8]:
#| export
ARPABET_VOWELS = {  # -> misaki's US phonemes, (stressed, unstressed)
    "AA": ("ɑ", "ɑ"),
    "AE": ("æ", "æ"),
    "AH": ("ʌ", "ə"),
    "AO": ("ɔ", "ɔ"),
    "AW": ("W", "W"),
    "AY": ("I", "I"),
    "EH": ("ɛ", "ɛ"),
    "ER": ("ɜɹ", "əɹ"),
    "EY": ("A", "A"),
    "IH": ("ɪ", "ɪ"),
    "IY": ("i", "i"),
    "OW": ("O", "O"),
    "OY": ("Y", "Y"),
    "UH": ("ʊ", "ʊ"),
    "UW": ("u", "u"),
}
ARPABET_CONSONANTS = {
    "B": "b",
    "CH": "ʧ",
    "D": "d",
    "DH": "ð",
    "F": "f",
    "G": "ɡ",
    "HH": "h",
    "JH": "ʤ",
    "K": "k",
    "L": "l",
    "M": "m",
    "N": "n",
    "NG": "ŋ",
    "P": "p",
    "R": "ɹ",
    "S": "s",
    "SH": "ʃ",
    "T": "t",
    "TH": "θ",
    "V": "v",
    "W": "w",
    "Y": "j",
    "Z": "z",
    "ZH": "ʒ",
}


def arpabet_to_misaki(arpa):
    "One CMUdict pronunciation in misaki's US phonemes, with its flapped t and syllabic final l, n and m."
    out = []
    for p in arpa.split():
        base, stress = re.fullmatch(r"([A-Z]+)([012])?", p).groups()
        if base in ARPABET_VOWELS:
            out.append({"1": "ˈ", "2": "ˌ"}.get(stress, "") + ARPABET_VOWELS[base][stress == "0"])
        else:
            out.append(ARPABET_CONSONANTS[base])
    s = re.sub(r"ə([lnm])$", r"ᵊ\1", "".join(out))
    return re.sub(r"([ɑæʌɔWIɛAɪiOYʊuəɜ]ɹ?)t(?=[əɪiOAʊuᵻ])", r"\1ɾ", s)


def cmudict():
    "CMUdict's first pronunciation of each word, in misaki's phonemes."
    path = G2P / "cmudict.dict"
    if not path.exists():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/cmusphinx/cmudict/master/cmudict.dict", path)
    out = {}
    for line in path.read_text().splitlines():
        word, arpa = line.split("#")[0].strip().split(" ", 1)
        if "(" not in word:
            out[word] = arpabet_to_misaki(arpa)
    return out


def misaki_lexicon(accent):
    "misaki's gold and silver lexicons as {lowercase word: phonemes}: gold first, lowercase entries before capitalised."
    out = {}
    for tier in ["gold", "silver"]:
        entries = json.loads(files("misaki.data").joinpath(f"{accent}_{tier}.json").read_text())
        for w, p in sorted(entries.items(), key=lambda kv: kv[0] != kv[0].lower()):
            p = p.get("DEFAULT", next(iter(p.values()))) if isinstance(p, dict) else p
            if p and re.fullmatch(r"[a-z]+(?:'[a-z]+)?", w.lower()) and w.lower() not in out:
                out[w.lower()] = p
    return out

In [9]:
cmu = cmudict()
strip = lambda s: re.sub("[ˈˌ]", "", s)
for accent in ["us", "gb"]:
    gold = json.loads(files("misaki.data").joinpath(f"{accent}_gold.json").read_text())
    shared = sorted(w for w, p in gold.items() if isinstance(p, str) and w.islower() and w.isalpha() and w in cmu)
    random.seed(0)
    sample = random.sample(shared, 3000)
    espeak = EspeakFallback(british=accent == "gb")
    sources = {"espeak-ng": {w: espeak(MToken(w, "NN", " "))[0] for w in sample}}
    if accent == "us":
        sources["CMUdict"] = cmu
    for name, src in sources.items():
        same = sum(src[w] == gold[w] for w in sample) / len(sample)
        loose = sum(strip(src[w]) == strip(gold[w]) for w in sample) / len(sample)
        print(f"{accent}, {name}: {same:.0%} the same as misaki's gold lexicon, {loose:.0%} ignoring stress")

us, espeak-ng: 43% the same as misaki's gold lexicon, 54% ignoring stress
us, CMUdict: 59% the same as misaki's gold lexicon, 65% ignoring stress


gb, espeak-ng: 57% the same as misaki's gold lexicon, 67% ignoring stress


Where they differ it's mostly reduced vowels (ə against ɪ) and secondary stress. So the US lexicon takes CMUdict
first and espeak for the rest; CMUdict is American, so the British one is all espeak. The lexicon also carries
misaki's letter names, for the last resort of spelling a word out.

In [10]:
#| export
def word_list():
    "Candidate words: CMUdict's, and macOS's word lists'."
    words = set(cmudict())
    for name in ["web2", "propernames"]:
        words |= {w.strip().lower() for w in Path(f"/usr/share/dict/{name}").read_text().splitlines()}
    return sorted(w for w in words if re.fullmatch(r"[a-z]+(?:'[a-z]+)?", w))


def unknown_words(words, accent):
    "The words misaki can't say on its own. A hundred to a call: misaki is slow per call, not per word."
    g = en.G2P(trf=False, british=accent == "gb", fallback=None)
    missing = set()
    for i in range(0, len(words), 100):
        missing |= {t.text.lower() for t in unknown(g(", ".join(words[i : i + 100]))[1])}
    return sorted(missing & set(words))


def build_lexicon(accent):
    "TEXT/lexicon_{accent}.json: {'letters': misaki's letter names, 'words': what misaki can't say}. Kept once made."
    path = TEXT / f"lexicon_{accent}.json"
    if path.exists():
        return json.loads(path.read_text())
    british, cmu = accent == "gb", cmudict()
    missing = unknown_words(word_list(), accent)
    espeak = EspeakFallback(british=british)
    batch = dict(zip(missing, espeak.backend.phonemize(missing)))  # One espeak call for all of them
    espeak.backend = type("Cached", (), {"phonemize": lambda self, ws: [batch[w] for w in ws]})()
    words = {}
    for w in missing:
        ps = cmu[w] if not british and w in cmu else espeak(MToken(w, "NN", " "))[0]
        if ps:
            words[w] = ps
    gold = json.loads(files("misaki.data").joinpath(f"{accent}_gold.json").read_text())
    lexicon = {"letters": {c.lower(): gold[c] for c in string.ascii_uppercase}, "words": words}
    path.write_text(json.dumps(lexicon, ensure_ascii=False, separators=(",", ":"), sort_keys=True))
    return lexicon

In [11]:
lexicons = {accent: build_lexicon(accent) for accent in ["us", "gb"]}
for accent, lex in lexicons.items():
    from_cmu = f" ({sum(lex['words'][w] == cmu.get(w) for w in lex['words'])} from CMUdict)" if accent == "us" else ""
    print(f"{accent}: {len(lex['words'])} words{from_cmu}, {size_mb(TEXT / f'lexicon_{accent}.json'):.1f} MB")
    print(
        "   ",
        ", ".join(
            f"{w} {lex['words'][w]}"
            for w in ["testified", "verified", "oswald", "kennedy", "clerkenwell"]
            if w in lex["words"]
        ),
    )

us: 188792 words (63965 from CMUdict), 5.7 MB
    testified tˈɛstəfˌId, verified vˈɛɹəfˌId, oswald ˈɔzwɔld
gb: 184095 words, 5.5 MB
    testified tˈɛstɪfˌId, verified vˈɛɹɪfˌId, oswald ˈɒswᵊld, kennedy kˈɛnədi


## Words in no lexicon

Names, brands and new words will still turn up that no list has: "Vimeo", "Datalab", "Mohrenschildt". Spelling
them out letter by letter is better than silence, but not by much. So the last resort before spelling is a small
model trained on everything above: misaki's lexicons and the new one, 360,000 words a voice.

It reads letters and writes phonemes, so it's a sequence-to-sequence problem, but a phone shouldn't have to run a
decoder loop for it. Instead: a transformer encoder over the letters, three output slots per letter (a phoneme
string runs up to about twice its spelling, and the stress marks are symbols too), trained with CTC, which lets
each slot output a phoneme or nothing. Decoding is one pass: the best symbol in each slot, repeats merged, blanks
dropped.

In [12]:
#| export
UP = 3  # Output slots per letter


class G2PNet(torch.nn.Module):
    "Letters (b, n) -> phoneme logits (b, 3n, phonemes): a pre-norm transformer encoder, decoded greedily as CTC."

    def __init__(self, letters, phonemes, d=256, layers=6, heads=4, max_len=UP * 32):
        super().__init__()
        self.embed = torch.nn.Embedding(letters, d, padding_idx=0)
        self.slot = torch.nn.Parameter(torch.randn(UP, d) * 0.02)  # Which of its letter's slots a position is
        pos = torch.arange(max_len)[:, None] / torch.pow(10_000, torch.arange(0, d, 2) / d)
        self.register_buffer("pos", torch.cat([torch.sin(pos), torch.cos(pos)], -1))
        layer = torch.nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.1, batch_first=True, norm_first=True)
        self.encoder = torch.nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.out = torch.nn.Sequential(torch.nn.LayerNorm(d), torch.nn.Linear(d, phonemes))

    def forward(self, letters, pad=None):
        x = self.embed(letters)
        b, n, d = x.shape
        x = (x[:, :, None] + self.slot).reshape(b, n * UP, d) + self.pos[: n * UP]
        return self.out(self.encoder(x, src_key_padding_mask=None if pad is None else pad.repeat_interleave(UP, 1)))


def ctc_decode(logits, n, phonemes):
    "The phonemes in the first 3n slots: the best symbol in each, repeats merged, blanks (0) dropped."
    out, prev = [], 0
    for i in logits[: UP * n].argmax(-1).tolist():
        if i != prev and i != 0:
            out.append(phonemes[i])
        prev = i
    return "".join(out)

The training data, split the same way every time so the held-out 2% stays held out:

In [13]:
#| export
def g2p_data(accent):
    "(lexicon, train words, test words, letters, phonemes) for one accent."
    lexicon = misaki_lexicon(accent)
    for w, p in build_lexicon(accent)["words"].items():
        lexicon.setdefault(w, p)
    words = sorted(w for w, p in lexicon.items() if len(w) <= 24 and len(p) <= UP * len(w))
    random.Random(0).shuffle(words)
    letters = ["<pad>"] + sorted({c for w in words for c in w})
    phonemes = ["<blank>"] + sorted({c for w in words for c in lexicon[w]})
    return lexicon, words[len(words) // 50 :], words[: len(words) // 50], letters, phonemes


def batches(words, letters, size, shuffle=False):
    "Words of similar length together: (the words, their letters as ids, zero-padded)."
    index = {c: i for i, c in enumerate(letters)}
    words = sorted(words, key=len)
    chunks = [words[i : i + size] for i in range(0, len(words), size)]
    if shuffle:
        random.shuffle(chunks)
    for chunk in chunks:
        x = torch.zeros(len(chunk), max(map(len, chunk)), dtype=torch.long)
        for i, w in enumerate(chunk):
            x[i, : len(w)] = torch.tensor([index[c] for c in w])
        yield chunk, x


def edit_distance(a, b):
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
    return row[-1]


@torch.no_grad()
def evaluate(net, words, lexicon, letters, phonemes, device="cpu"):
    "(share of words exactly right, phoneme error rate)."
    net.eval()
    right = errors = total = 0
    for chunk, x in batches(words, letters, 512):
        logits = net(x.to(device), (x == 0).to(device)).cpu()
        for i, w in enumerate(chunk):
            ps = ctc_decode(logits[i], len(w), phonemes)
            right += ps == lexicon[w]
            errors += edit_distance(ps, lexicon[w])
            total += len(lexicon[w])
    return right / len(words), errors / total

In [14]:
#| export
def train_g2p(accent, epochs=15, log=print):
    "Train G2PNet for one accent (an hour and a quarter on an M3 Pro), or load it once trained: G2P/g2p_{accent}.pt."
    path = G2P / f"g2p_{accent}.pt"
    lexicon, train, test, letters, phonemes = g2p_data(accent)
    net = G2PNet(len(letters), len(phonemes))
    if path.exists():
        net.load_state_dict(torch.load(path)["state"])
        return net.eval()
    torch.manual_seed(0)
    random.seed(0)
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    net = net.to(device)
    steps = epochs * math.ceil(len(train) / 256)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=steps, pct_start=0.05)
    index = {c: i for i, c in enumerate(phonemes)}
    t = time.time()
    for epoch in range(epochs):
        net.train()
        for chunk, x in batches(train, letters, 256, shuffle=True):
            targets = [torch.tensor([index[c] for c in lexicon[w]]) for w in chunk]
            logp = F.log_softmax(net(x.to(device), (x == 0).to(device)).float(), -1).transpose(0, 1).cpu()  # CTC: CPU
            loss = F.ctc_loss(
                logp,
                torch.cat(targets),
                torch.tensor([UP * len(w) for w in chunk]),
                torch.tensor([len(tg) for tg in targets]),
                zero_infinity=True,
            )
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            sched.step()
        if epoch % 5 == 4:
            right, per = evaluate(net, test, lexicon, letters, phonemes, device)
            log(
                f"epoch {epoch + 1}: {right:.1%} of test words right, {per:.2%} phoneme error rate, {(time.time() - t) / 60:.0f} min"
            )
    net = net.eval().cpu()
    torch.save({"state": net.state_dict(), "letters": letters, "phonemes": phonemes}, path)
    return net

In [15]:
nets = {accent: train_g2p(accent) for accent in ["us", "gb"]}
names = ["clerkenwell", "lovelady", "mohrenschildt", "vimeo", "datalab", "patreon", "thistlewood", "subiaco"]
for accent, net in nets.items():
    lexicon, _, test, letters, phonemes = g2p_data(accent)
    extra = [w for w in test if w in lexicons[accent]["words"]]
    right, per = evaluate(net, test, lexicon, letters, phonemes)
    right_extra, per_extra = evaluate(net, extra, lexicon, letters, phonemes)
    print(
        f"{accent}: {right:.0%} of held-out words exactly right ({per:.1%} phoneme errors); "
        f"of the rare words and names among them, {right_extra:.0%} ({per_extra:.1%})"
    )
    index = {c: i for i, c in enumerate(letters)}
    with torch.no_grad():
        print(
            "   ",
            "; ".join(
                f"{w} {ctc_decode(net(torch.tensor([[index[c] for c in w]]))[0], len(w), phonemes)}" for w in names
            ),
        )

us: 63% of held-out words exactly right (7.3% phoneme errors); of the rare words and names among them, 61% (7.3%)
    clerkenwell klˈɜɹkənwˌɛl; lovelady lˈʌvlAdi; mohrenschildt mˈɔɹənʃˌɪlt; vimeo vˈɪmiO; datalab dˈATᵊlˌæb; patreon pˈAtɹiən; thistlewood θˈɪsᵊlwˌʊd; subiaco subiˈɑkO


gb: 75% of held-out words exactly right (5.2% phoneme errors); of the rare words and names among them, 77% (4.0%)
    clerkenwell klˈɑːkənwˌɛl; lovelady lˈʌvlAdi; mohrenschildt mˈQɹənsˌIlt; vimeo vˈɪmɪˌQ; datalab dˈAtəlˌab; patreon pˈatɹɪən; thistlewood θˈɪsᵊlwˌʊd; subiaco sˌʌbɪˈɑːkQ


The American model does worse than the British one, probably because its lexicon mixes two sources' conventions,
CMUdict's and espeak's, for the model to reconcile. Most of its misses are stress marks and reduced vowels, and every
name above is said the way a reader would guess it.

The apps run it in plain Rust, inside the same library as misaki, so it needs no runtime of its own. Its weights go
out as 16-bit floats behind a JSON header:

In [16]:
#| export
def export_g2p(accent):
    "TEXT/g2p_{accent}.g2p: 'G2P1', a little-endian u32 header length, a JSON header, then the weights as 16-bit floats."
    c = torch.load(G2P / f"g2p_{accent}.pt")
    tensors, chunks, offset = {}, [], 0
    for name, t in c["state"].items():
        a = t.float().numpy().astype("<f2")
        tensors[name] = [list(a.shape), offset]
        chunks.append(a.tobytes())
        offset += a.size
    layers = 1 + max(int(n.split(".")[2]) for n in tensors if n.startswith("encoder.layers."))
    header = json.dumps(
        {
            "letters": c["letters"],
            "phonemes": c["phonemes"],
            "up": UP,
            "heads": 4,
            "layers": layers,
            "tensors": tensors,
        },
        ensure_ascii=False,
    ).encode()
    with open(TEXT / f"g2p_{accent}.g2p", "wb") as f:
        f.write(b"G2P1" + struct.pack("<I", len(header)) + header + b"".join(chunks))

In [17]:
for accent in ["us", "gb"]:
    export_g2p(accent)
    print(f"g2p_{accent}.g2p: {size_mb(TEXT / f'g2p_{accent}.g2p'):.1f} MB")

g2p_us.g2p: 9.6 MB
g2p_gb.g2p: 9.6 MB


## misaki in Rust

misaki-rs is an MIT-licensed port of misaki 0.9.4, the version Kokoro was trained with, including the parts of
spaCy misaki depends on: its tokenizer and tagger, the tagger's weights and all. audio-reader vendors it in
`native/g2p`, changed to read its lexicons and tagger from the download rather than build 20 MB of them into the app,
and wraps it in `Phonemizer`, which adds the lexicon and model above as its fallback. The data comes from the
misaki-rs release:

In [18]:
#| export
MISAKI_RS = "0.8.0"
MISAKI_RS_DATA = ["us_gold.json", "us_silver.json", "gb_gold.json", "gb_silver.json", "tagger.bin", "tagger_labels.txt"]


def misaki_rs_data():
    "misaki-rs's lexicons and tagger, from its crates.io release, into TEXT."
    if all((TEXT / f).exists() for f in MISAKI_RS_DATA):
        return
    url = f"https://crates.io/api/v1/crates/misaki-rs/{MISAKI_RS}/download"
    request = urllib.request.Request(url, headers={"User-Agent": "text-to-speech notebooks"})
    with urllib.request.urlopen(request) as r, tarfile.open(fileobj=r, mode="r|gz") as tar:
        for member in tar:
            name = member.name.split("/")[-1]
            if member.isfile() and member.name.endswith(f"/data/{name}") and name in MISAKI_RS_DATA:
                (TEXT / name).write_bytes(tar.extractfile(member).read())


@functools.cache
def rust_example(name):
    "Build one of native/g2p's examples, and return its path."
    target = NATIVE / "build" / "target"
    subprocess.run(
        ["rustup", "run", "1.97", "cargo", "build", "--release", "--example", name],
        cwd=NATIVE,
        check=True,
        capture_output=True,
        env={**os.environ, "CARGO_TARGET_DIR": str(target)},
    )
    return target / "release" / "examples" / name


def rust_g2p(lines, accent):
    "native/g2p's phonemes for each line: {'phonemes', 'tokens': [{'text', 'phonemes', 'whitespace', 'start', 'end'}]}."
    r = subprocess.run(
        [str(rust_example("phonemize")), str(TEXT), accent],
        input="\n".join(lines) + "\n",
        capture_output=True,
        text=True,
        check=True,
    )
    return [json.loads(line) for line in r.stdout.splitlines()]

In [19]:
misaki_rs_data()
t = time.perf_counter()
rust = {accent: rust_g2p(lines, accent) for accent in ["us", "gb"]}
print(f"{2 * len(lines)} lines in {time.perf_counter() - t:.0f} s")


def rewritten(line, r):
    "Whether native/g2p wrote out an abbreviation in this line: a token whose text isn't what it points at."
    u16 = line.encode("utf-16-le")
    return any(u16[2 * t["start"] : 2 * t["end"]].decode("utf-16-le") != t["text"] for t in r["tokens"])


for accent in ["us", "gb"]:
    rows = [(p, r) for line, p, r in zip(lines, python[accent], rust[accent]) if not unknown(p[1])]
    clean = [
        (p, r)
        for (p, r), line in zip(rows, [l for l, p in zip(lines, python[accent]) if not unknown(p[1])])
        if not rewritten(line, r)
    ]
    kokoro = lambda ps: (ps or "").replace("❓", "")  # As Kokoro reads it: ❓ isn't in its vocabulary
    same = sum(kokoro(p[0]) == kokoro(r["phonemes"]) for p, r in clean)
    pairs = sum(
        [[t.text, kokoro(t.phonemes)] for t in p[1]] == [[t["text"], kokoro(t["phonemes"])] for t in r["tokens"]]
        for p, r in clean
    )
    print(
        f"{accent}: of {len(clean)} lines misaki can say in full ({len(rows) - len(clean)} more have abbreviations "
        f"written out), {same} identical phonemes, {pairs} identical tokens"
    )

3948 lines in 60 s
us: of 1474 lines misaki can say in full (3 more have abbreviations written out), 1471 identical phonemes, 1471 identical tokens
gb: of 1463 lines misaki can say in full (3 more have abbreviations written out), 1460 identical phonemes, 1460 identical tokens


Identical, as Kokoro reads them, on all but a few of the lines misaki can say in full, token by token, which matters
because each token's phonemes are what its word's timing comes from. (The lines with abbreviations differ on purpose:
more on them below.) The rest:

In [20]:
import difflib

for (p, r), line in [(pr, l) for pr, l in zip(zip(python["us"], rust["us"]), lines) if not unknown(pr[0][1])]:
    if p[0].replace("❓", "") != r["phonemes"].replace("❓", "") and not rewritten(line, r):
        a, b = p[0].replace("❓", "").split(), r["phonemes"].replace("❓", "").split()
        print(" ".join(f"{x[0]}{x[2:]}" for x in difflib.ndiff(a, b) if x[0] in "+-")[:110])

+pYnt
+pYnt
+pYnt


A decimal inside a word: misaki reads "98.7%" as "ninety eight seven percent", and the Rust says "point", on purpose.
The vendored copy fixes four other differences, each with a test there: a symbol misaki can't say joining the number
before it rather than standing apart, "3's" as "threes", "$0.042" as a number rather than cents, and the "zero" of
"3.0". Lines with unknown words now get the fallback:

In [21]:
for accent in ["us", "gb"]:
    words = lexicons[accent]["words"]
    from_lexicon, from_model = 0, {}
    for (_, tokens), r in zip(python[accent], rust[accent]):
        for t in unknown(tokens):
            out = next((x for x in r["tokens"] if x["text"] == t.text), None)
            if out is None:
                continue
            if t.text.lower().strip(".'’s") in words:
                from_lexicon += 1
            else:
                from_model[t.text] = out["phonemes"]
    print(f"{accent}: {from_lexicon} unknown words from the lexicon; {len(from_model)} distinct from the model, e.g.")
    print("   ", "; ".join(f"{w} {p}" for w, p in list(from_model.items())[:12]))

us: 394 unknown words from the lexicon; 222 distinct from the model, e.g.
    Cuvier kjˈʌviəɹ; roominghouse ɹˈumɪŋhˌWs; wardsmen wˈɔɹdzmɛn; Euins jˈuɪnz; Hidell hɪdˈᵊl; Mohrenschildts mˈɔɹənʃˌɪlts; palmprint pˈɑlmpɹɪnt; piemen pˈimɛn; Mohrenschildt mˈɔɹənʃˌɪlt; Simmons sˈɪmənz; McWatters' məkwˈɔTəɹz; Jumjuma ʤˈʌmʤˈumə
gb: 418 unknown words from the lexicon; 219 distinct from the model, e.g.
    Cuvier kjˈuːvɪə; roominghouse ɹˈuːmɪŋhˌWs; wardsmen wˈɔːdzmɛn; Anti-Kennedy ˈanti kˈɛnədi; Euins jˈuːɪnz; Hidell hˈIdᵊl; Mohrenschildts mˈQɹənsˌIlts; palmprint pˈɑːmpɹɪnt; piemen pˈiːmɛn; Mohrenschildt mˈQɹənsˌIlt; Simmons sˈɪmənz; McWatters' məkwˈɒtəz


The Rust model is the PyTorch one with 16-bit weights:

In [22]:
words = random.Random(3).sample(sorted(lexicons["gb"]["words"]), 2000)
_, _, _, letters, phonemes = g2p_data("gb")
index = {c: i for i, c in enumerate(letters)}
with torch.no_grad():
    torch_out = [ctc_decode(nets["gb"](torch.tensor([[index[c] for c in w]]))[0], len(w), phonemes) for w in words]
t = time.perf_counter()
r = subprocess.run(
    [str(rust_example("g2p_net")), str(TEXT / "g2p_gb.g2p")],
    input="\n".join(words) + "\n",
    capture_output=True,
    text=True,
    check=True,
)
elapsed = time.perf_counter() - t
rust_out = [line.split("\t")[1] for line in r.stdout.splitlines()]
print(
    f"{sum(a == b for a, b in zip(torch_out, rust_out))} of {len(words)} words the same; {elapsed / len(words) * 1000:.0f} ms a word"
)

2000 of 2000 words the same; 70 ms a word


## Abbreviations

misaki says "Mr.", "Dr.", "No." and "vs." well, and spells out or garbles plenty more: "St." as "S T", "Gov." as
"G O V", "km" as "kkm". `native/g2p/src/normalize.rs` writes them out before misaki sees them, by context where the
spelling isn't enough ("St. Paul's" is Saint, "Baker St." Street; a unit only after a number), and maps every token
back to the characters it came from, so the app can still highlight "St." while saying "Street":

In [23]:
survey = [
    "St. Paul's Cathedral is on Ludgate Hill, not Baker St. in London.",
    "Gen. Patton, Gov. Newsom and Capt. Kirk met John Smith Jr. at 9 a.m. on Mon. 3 Jan.",
    "Widgets Inc. and Gadgets Ltd. moved approx. 5 km, viz. from No. 10 Downing St. to Mt. Pleasant.",
    "See Fig. 3, Vol. 2, pp. 14-20, ed. by J. R. R. Tolkien; it weighs 3 kg and is 6 ft tall.",
]
for line, r in zip(survey, rust_g2p(survey, "gb")):
    u16 = line.encode("utf-16-le")
    print(
        " ".join(
            f"{u16[2 * t['start'] : 2 * t['end']].decode('utf-16-le')}→{t['text']}"
            for t in r["tokens"]
            if u16[2 * t["start"] : 2 * t["end"]].decode("utf-16-le") != t["text"] and t["text"].strip(".,")
        )
    )

St.→Saint St.→Street
Gen.→General Gov.→Governor Capt.→Captain Jr.→Junior Mon.→Monday Jan.→January
Inc.→Incorporated Ltd.→Limited approx.→approximately km→kilometres viz.→namely St.→Street Mt.→Mount
Fig.→Figure Vol.→Volume pp.→pages ed.→edited kg→kilograms ft→feet


## What each app downloads

One folder per platform: notebook 106's models and speakers, these text files, Kokoro's phoneme vocabulary, and a
`voice.json` saying which sizes the body comes in: all six on iOS, which loads two of them, and three on Android. audio-reader's `scripts/voice-pack.sh` packs the iOS one as an
asset pack. The simulator's Core ML won't load the shared-weight body, so its folder carries one body per size too.

Android loads every size in its body file onto the GPU at once, and each size keeps its own working memory
there: on a Pixel 10a, 539 MB for all six against 307 MB for 128, 256 and 512 frames, with the weights shared either
way. Dropping 192 and 384 pads those sentences to the next size up, which costs some speed: they still render at 2.8
times real time or better, and short sentences are as quick as before. 64 frames never came up, even for "Good
morning." Two sizes, as iOS uses, cost
too much: a sentence just over 192 frames took 512 and fell to 1.9 times real time, below the 2× the reader listens at.
So the Android folder keeps three. Each signature's subgraph is kept whole, with the same shared weights, so what
they compute is unchanged bit for bit.

In [ ]:
#| export
ANDROID_SIZES = [128, 256, 512]  # Frames. See above: the GPU's memory goes with how many sizes there are

TRIM_SIZES = """
import sys, flatbuffers, numpy as np
from ai_edge_litert import schema_py_generated as s
src, dst, keep = sys.argv[1], sys.argv[2], [f"body{int(T)}" for T in sys.argv[3:]]
raw = open(src, "rb").read()
m = s.ModelT.InitFromObj(s.Model.GetRootAsModel(raw, 0))
for buf in m.buffers:  # The converter stores weights after the flatbuffer, by offset: bring them inline
    if buf.offset and buf.offset > 1:
        buf.data = np.frombuffer(raw, dtype=np.uint8, count=buf.size, offset=buf.offset)
        buf.offset = buf.size = 0
calls = {s.BuiltinOperator.CALL_ONCE, s.BuiltinOperator.WHILE, s.BuiltinOperator.IF}
for g in m.subgraphs:  # Subgraphs calling others would need their indices remapped
    for op in g.operators:
        code = m.operatorCodes[op.opcodeIndex]
        assert max(code.builtinCode, code.deprecatedBuiltinCode) not in calls
defs = [d for d in m.signatureDefs if d.signatureKey.decode() in keep]
assert len(defs) == len(keep), "missing sizes"
m.subgraphs = [m.subgraphs[d.subgraphIndex] for d in defs]
for i, d in enumerate(defs):
    d.subgraphIndex = i
m.signatureDefs = defs
used = sorted({t.buffer for g in m.subgraphs for t in g.tensors} | {0})  # Buffer 0 is always the empty one
index = {b: i for i, b in enumerate(used)}
m.buffers = [m.buffers[b] for b in used]
for g in m.subgraphs:
    for t in g.tensors:
        t.buffer = index[t.buffer]
m.metadata = [md for md in m.metadata or [] if md.buffer in index]
for md in m.metadata:
    md.buffer = index[md.buffer]
b = flatbuffers.Builder(1 << 20)
b.Finish(m.Pack(b), file_identifier=b"TFL3")
open(dst, "wb").write(b.Output())
"""


def trim_tflite(src, dst, sizes):
    "A copy of the LiteRT body at `src` with only the signatures for `sizes`, in `dst`."
    uv = shutil.which("uv") or os.path.expanduser("~/.local/bin/uv")
    env = [uv, "run", "-q", "--no-project", "--python", "3.13", "--with", "ai-edge-litert", "--with", "flatbuffers"]
    subprocess.run([*env, "python", "-c", TRIM_SIZES, str(src), str(dst), *map(str, sizes)], check=True)


In [24]:
#| export
def voice_folder(platform, simulator=False):
    "Everything the app on `platform` ('ios' or 'android') downloads, in VOICE/<platform>-pack. Returns the folder."
    out = VOICE / f"{platform}-pack"
    shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True)
    source, models = (
        (
            IOS,
            {
                "tokens.mlmodelc": "tokens8.mlmodelc",
                "prosody.mlmodelc": "prosody8.mlmodelc",
                "source.mlmodelc": "source.mlmodelc",
                "body.mlmodelc": "body8.mlmodelc",
            },
        )
        if platform == "ios"
        else (
            ANDROID,
            {
                "tokens.onnx": "tokens8.onnx",
                "prosody.onnx": "prosody8.onnx",
                "source.onnx": "source.onnx",
            },
        )
    )
    for name, built in models.items():
        (shutil.copytree if (source / built).is_dir() else shutil.copy)(source / built, out / name)
    shutil.copytree(source / "voices", out / "voices")
    for f in TEXT.iterdir():
        shutil.copy(f, out / f.name)
    (out / "vocab.json").write_text(json.dumps(model.vocab, ensure_ascii=False))
    sizes = SIZES if platform == "ios" else ANDROID_SIZES
    if platform == "android":
        trim_tflite(ANDROID / "body_fp16.tflite", out / "body.tflite", sizes)
    (out / "voice.json").write_text(json.dumps({"sizes": sizes, "sampleRate": SR}))
    if simulator:
        import coremltools as ct

        for T in SIZES:
            ct.utils.compile_model(str(VOICE / "parts" / f"body8{T}.mlpackage"), str(out / f"body{T}.mlmodelc"))
    return out

In [25]:
for platform in ["ios", "android"]:
    folder = voice_folder(platform)
    print(
        f"{platform}: {size_mb(folder):.0f} MB:",
        ", ".join(f"{f.name} {size_mb(f):.1f}" for f in sorted(folder.iterdir())),
    )

ios: 181 MB: body.mlmodelc 55.7, g2p_gb.g2p 9.6, g2p_us.g2p 9.6, gb_gold.json 2.8, gb_silver.json 3.7, lexicon_gb.json 5.5, lexicon_us.json 5.7, prosody.mlmodelc 9.2, source.mlmodelc 0.0, tagger.bin 6.3, tagger_labels.txt 0.0, tokens.mlmodelc 66.1, us_gold.json 3.0, us_silver.json 3.1, vocab.json 0.0, voice.json 0.0, voices 1.0


android: 240 MB: body.tflite 119.7, g2p_gb.g2p 9.6, g2p_us.g2p 9.6, gb_gold.json 2.8, gb_silver.json 3.7, lexicon_gb.json 5.5, lexicon_us.json 5.7, prosody.onnx 8.6, source.onnx 0.0, tagger.bin 6.3, tagger_labels.txt 0.0, tokens.onnx 61.8, us_gold.json 3.0, us_silver.json 3.1, vocab.json 0.0, voice.json 0.0, voices 1.0


## Summary

- **misaki can't say** about one word in seventy in this corpus, mostly names. Without espeak-ng, each would be
  skipped.
- **A lexicon built in advance** covers 190,000 such words a voice, from CMUdict and espeak-ng run here, once. The
  apps never contain espeak.
- **A small model** (4.8M parameters, CTC over three slots a letter) guesses the rest: 75% of held-out British words
  exactly right, 63% American, and names come out the way a reader would say them. It runs in plain Rust.
- **misaki-rs**, vendored in audio-reader, says what Python misaki says, token for token, on all but three lines it
  can say in full; those three gain a "point" in a decimal, on purpose.
- **Abbreviations** are written out by context before misaki sees them, with every token still pointing at its
  original characters.